In [1]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

In [2]:
train_df = pd.read_csv("dataset/raw/train.csv")
test_df  = pd.read_csv("dataset/raw/test.csv")

In [104]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200000 entries, 0 to 1199999
Data columns (total 21 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   id                    1200000 non-null  int64  
 1   Age                   1181295 non-null  float64
 2   Gender                1200000 non-null  object 
 3   Annual Income         1155051 non-null  float64
 4   Marital Status        1181471 non-null  object 
 5   Number of Dependents  1090328 non-null  float64
 6   Education Level       1200000 non-null  object 
 7   Occupation            841925 non-null   object 
 8   Health Score          1125924 non-null  float64
 9   Location              1200000 non-null  object 
 10  Policy Type           1200000 non-null  object 
 11  Previous Claims       835971 non-null   float64
 12  Vehicle Age           1199994 non-null  float64
 13  Credit Score          1062118 non-null  float64
 14  Insurance Duration    1199999 non-

In [105]:
train_df.head()

,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,...,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type,Premium Amount
0,0,19.0,Female,10049.0,Married,1.0,Bachelor's,Self-Employed,22.598761,Urban,...,2.0,17.0,372.0,5.0,2023-12-23 15:21:39.134960,Poor,No,Weekly,House,2869.0
1,1,39.0,Female,31678.0,Divorced,3.0,Master's,NaN,15.569731,Rural,...,1.0,12.0,694.0,2.0,2023-06-12 15:21:39.111551,Average,Yes,Monthly,House,1483.0
2,2,23.0,Male,25602.0,Divorced,3.0,High School,Self-Employed,47.177549,Suburban,...,1.0,14.0,NaN,3.0,2023-09-30 15:21:39.221386,Good,Yes,Weekly,House,567.0
3,3,21.0,Male,141855.0,Married,2.0,Bachelor's,NaN,10.938144,Rural,...,1.0,0.0,367.0,1.0,2024-06-12 15:21:39.226954,Poor,Yes,Daily,Apartment,765.0
4,4,21.0,Male,39651.0,Single,1.0,Bachelor's,Self-Employed,20.376094,Rural,...,0.0,8.0,598.0,4.0,2021-12-01 15:21:39.252145,Poor,Yes,Weekly,House,2022.0


In [106]:
train_df.isnull().sum()

id                           0
Age                      18705
Gender                       0
Annual Income            44949
Marital Status           18529
Number of Dependents    109672
Education Level              0
Occupation              358075
Health Score             74076
Location                     0
Policy Type                  0
Previous Claims         364029
Vehicle Age                  6
Credit Score            137882
Insurance Duration           1
Policy Start Date            0
Customer Feedback        77824
Smoking Status               0
Exercise Frequency           0
Property Type                0
Premium Amount               0
dtype: int64

In [107]:
def show_unique_values(df, columns=None, max_display=50):
    """
    Hiển thị các giá trị unique của các cột trong DataFrame
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame cần phân tích
    columns : list, optional
        Danh sách các cột cần xem. Nếu None, sẽ hiển thị tất cả các cột
    max_display : int, default=50
        Số lượng unique values tối đa hiển thị cho mỗi cột
        
    Returns:
    --------
    dict
        Dictionary chứa unique values của từng cột
    """
    # Nếu không chỉ định columns, lấy tất cả
    if columns is None:
        columns = df.columns.tolist()
    
    # Dictionary để lưu kết quả
    unique_dict = {}
    
    print("=" * 100)
    print("UNIQUE VALUES ANALYSIS")
    print("=" * 100)
    
    for col in columns:
        if col not in df.columns:
            print(f"Cột '{col}' không tồn tại trong DataFrame")
            continue
        
        unique_vals = df[col].unique()
        n_unique = len(unique_vals)
        
        # Lấy value counts
        value_counts = df[col].value_counts(dropna=False)
        
        # Lưu vào dictionary
        unique_dict[col] = unique_vals
        
        # In thông tin
        print(f"\n📊 Feature: {col}")
        print(f"   • Data type: {df[col].dtype}")
        print(f"   • Total unique values: {n_unique}")
        print(f"   • Missing values: {df[col].isnull().sum()} ({df[col].isnull().sum()/len(df)*100:.2f}%)")
        
        # Hiển thị unique values
        if n_unique <= max_display:
            print(f"   • Unique values: {sorted([str(v) for v in unique_vals if pd.notna(v)])}")
            print(f"\n   Value Counts:")
            for val, count in value_counts.head(20).items():
                pct = (count / len(df)) * 100
                print(f"      {str(val):<30} : {count:>6} ({pct:>5.2f}%)")
        else:
            print(f"   • Too many unique values ({n_unique}). Showing top 20:")
            for val, count in value_counts.head(20).items():
                pct = (count / len(df)) * 100
                print(f"      {str(val):<30} : {count:>6} ({pct:>5.2f}%)")
        
        print("-" * 100)
    
    return unique_dict

In [108]:
# Ví dụ 3: Xem unique values của tất cả các cột (cẩn thận với dataset lớn)
unique_all = show_unique_values(train_df)

UNIQUE VALUES ANALYSIS

📊 Feature: id
   • Data type: int64
   • Total unique values: 1200000
   • Missing values: 0 (0.00%)
   • Too many unique values (1200000). Showing top 20:
      0                              :      1 ( 0.00%)
      1                              :      1 ( 0.00%)
      2                              :      1 ( 0.00%)
      3                              :      1 ( 0.00%)
      4                              :      1 ( 0.00%)
      5                              :      1 ( 0.00%)
      6                              :      1 ( 0.00%)
      7                              :      1 ( 0.00%)
      8                              :      1 ( 0.00%)
      9                              :      1 ( 0.00%)
      10                             :      1 ( 0.00%)
      11                             :      1 ( 0.00%)
      12                             :      1 ( 0.00%)
      13                             :      1 ( 0.00%)
      14                             :      1 ( 0.

## Basic Processing

In [109]:
def convert_float_to_int(df, column_name):
    """
    Chuyển đổi cột từ float sang Int64 (nullable integer)
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame chứa cột cần chuyển đổi
    column_name : str
        Tên cột cần chuyển đổi
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame với cột đã được chuyển đổi
    """
    # Kiểm tra xem cột có tồn tại không
    if column_name not in df.columns:
        print(f"Cột '{column_name}' không tồn tại trong DataFrame")
        return df
    
    # Kiểm tra kiểu dữ liệu hiện tại
    current_dtype = df[column_name].dtype
    print('\n')
    print(f"📊 Kiểu dữ liệu hiện tại của '{column_name}': {current_dtype}")
    
    # Kiểm tra số lượng missing values
    null_count = df[column_name].isnull().sum()
    
    if null_count > 0:
        print(f"Có {null_count} giá trị null trong cột '{column_name}'")
    
    # Chuyển sang Int64 (hỗ trợ cả null)
    df[column_name] = df[column_name].astype('Int64')
    print(f"Đã chuyển '{column_name}' sang kiểu: {df[column_name].dtype}")
    
    return df

In [110]:
columns_change_type = ['Age', 'Number of Dependents', 'Annual Income', 'Previous Claims', 'Vehicle Age', 'Credit Score', 'Insurance Duration']

In [111]:
for col in columns_change_type:
    convert_float_to_int(train_df, col)



📊 Kiểu dữ liệu hiện tại của 'Age': float64
Có 18705 giá trị null trong cột 'Age'
Đã chuyển 'Age' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Number of Dependents': float64
Có 109672 giá trị null trong cột 'Number of Dependents'
Đã chuyển 'Number of Dependents' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Annual Income': float64
Có 44949 giá trị null trong cột 'Annual Income'
Đã chuyển 'Annual Income' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Previous Claims': float64
Có 364029 giá trị null trong cột 'Previous Claims'
Đã chuyển 'Previous Claims' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Vehicle Age': float64
Có 6 giá trị null trong cột 'Vehicle Age'
Đã chuyển 'Vehicle Age' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Credit Score': float64
Có 137882 giá trị null trong cột 'Credit Score'
Đã chuyển 'Credit Score' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Insurance Duration': float64
Có 1 giá trị null trong cột 'Insurance Duration'
Đã chuyển 'Insurance Duratio

### convert column in test_df

In [112]:
for col in columns_change_type:
    convert_float_to_int(test_df, col)



📊 Kiểu dữ liệu hiện tại của 'Age': float64
Có 12489 giá trị null trong cột 'Age'
Đã chuyển 'Age' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Number of Dependents': float64
Có 73130 giá trị null trong cột 'Number of Dependents'
Đã chuyển 'Number of Dependents' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Annual Income': float64
Có 29860 giá trị null trong cột 'Annual Income'
Đã chuyển 'Annual Income' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Previous Claims': float64
Có 242802 giá trị null trong cột 'Previous Claims'
Đã chuyển 'Previous Claims' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Vehicle Age': float64
Có 3 giá trị null trong cột 'Vehicle Age'
Đã chuyển 'Vehicle Age' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Credit Score': float64
Có 91451 giá trị null trong cột 'Credit Score'
Đã chuyển 'Credit Score' sang kiểu: Int64


📊 Kiểu dữ liệu hiện tại của 'Insurance Duration': float64
Có 2 giá trị null trong cột 'Insurance Duration'
Đã chuyển 'Insurance Duration'

### Convert "Policy Start Date" object datatype into Datetime datatype

In [113]:
train_df["Policy Start Date"] = pd.to_datetime(train_df['Policy Start Date'], errors='ignore')

C:\Users\dangn\AppData\Local\Temp\ipykernel_12872\704635338.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  train_df["Policy Start Date"] = pd.to_datetime(train_df['Policy Start Date'], errors='ignore')


In [114]:
test_df["Policy Start Date"] = pd.to_datetime(test_df['Policy Start Date'], errors='ignore')

C:\Users\dangn\AppData\Local\Temp\ipykernel_12872\2883126432.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  test_df["Policy Start Date"] = pd.to_datetime(test_df['Policy Start Date'], errors='ignore')


### Train Test Split

In [115]:
X_train = train_df.drop("Premium Amount", axis=1)
y_train = train_df["Premium Amount"]
X_test = test_df.copy()

### Xác định các loại biến (categorical vs numerical)

In [116]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = [col for col in X_train.select_dtypes(include=['object', 'category']).columns.tolist()]

categorical_ordinal = ['Education Level', 'Policy Type', 'Customer Feedback', 'Exercise Frequency']
categorical_nominal = [col for col in categorical_features if col not in categorical_ordinal]

print(f"📊 Số biến numerical: {len(numeric_features)}")
print(f"   Danh sách: {numeric_features}\n")
print(f"📊 Số biến categorical: {len(categorical_features)}")
print(f"   Danh sách: {categorical_features}\n")
print(f"📊 Số biến categorical ordinal: {len(categorical_ordinal)}")
print(f"   Danh sách: {categorical_ordinal}\n")
print(f"📊 Số biến categorical nominal: {len(categorical_nominal)}")
print(f"   Danh sách: {categorical_nominal}")

📊 Số biến numerical: 9
   Danh sách: ['id', 'Age', 'Annual Income', 'Number of Dependents', 'Health Score', 'Previous Claims', 'Vehicle Age', 'Credit Score', 'Insurance Duration']

📊 Số biến categorical: 10
   Danh sách: ['Gender', 'Marital Status', 'Education Level', 'Occupation', 'Location', 'Policy Type', 'Customer Feedback', 'Smoking Status', 'Exercise Frequency', 'Property Type']

📊 Số biến categorical ordinal: 4
   Danh sách: ['Education Level', 'Policy Type', 'Customer Feedback', 'Exercise Frequency']

📊 Số biến categorical nominal: 6
   Danh sách: ['Gender', 'Marital Status', 'Occupation', 'Location', 'Smoking Status', 'Property Type']


### Remove Zero Variance Feature and duplacted row

In [117]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline

class VarianceThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0):
        self.threshold = threshold
        self.selector = VarianceThreshold(threshold=self.threshold)
    
    def fit(self, X, y=None):
        numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns
        self.numeric_cols = numeric_cols
        self.selector.fit(X[numeric_cols])
        # Lưu lại cột số được giữ lại
        self.retained_cols = numeric_cols[self.selector.get_support()]
        # Lưu lại cột số bị loại
        self.dropped_cols = [col for col in numeric_cols if col not in self.retained_cols]
        print("Các cột bị loại vì phương sai = 0:", self.dropped_cols)
        return self
    
    def transform(self, X):
        cols_to_keep = list(self.retained_cols) + list(X.columns.difference(self.numeric_cols))
        return X[cols_to_keep]


In [118]:
class ConstantAndDuplicateRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Xác định cột hằng số
        self.constant_cols = [col for col in X.columns if X[col].nunique() == 1]
        print("Các cột bị loại vì chỉ có 1 giá trị:", self.constant_cols)
        return self
    
    def transform(self, X):
        # Bỏ cột hằng số
        X_cleaned = X.drop(columns=self.constant_cols, errors="ignore")
        # Bỏ dòng trùng lặp (không in số lượng nữa)
        X_cleaned = X_cleaned.drop_duplicates()
        return X_cleaned

In [119]:
cleaning_pipeline = Pipeline(steps=[
    ("var_thresh", VarianceThresholdSelector(threshold=0)),
    ("const_dup", ConstantAndDuplicateRemover())

])
X_train_cleaned = cleaning_pipeline.fit_transform(X_train.copy())
print(f"Kích thước dữ liệu sau khi làm sạch: {X_train_cleaned.shape}")
X_test_cleaned = cleaning_pipeline.transform(X_test.copy())
print(f"Kích thước dữ liệu sau khi làm sạch: {X_test_cleaned.shape}")

Các cột bị loại vì phương sai = 0: []
Các cột bị loại vì chỉ có 1 giá trị: []
Kích thước dữ liệu sau khi làm sạch: (1200000, 20)
Kích thước dữ liệu sau khi làm sạch: (800000, 20)


c:\Users\dangn\.conda\envs\dpav_env\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


# Missing processing

In [120]:
# Phân tích chi tiết theo nhóm
print("=" * 90)
print("DETAILED MISSING VALUES ANALYSIS")
print("=" * 90)

# Xác định loại biến
def get_feature_type(col):
    if col in numeric_features:
        return 'Numerical'
    elif col in categorical_ordinal:
        return 'Cat-Ordinal'
    elif col in categorical_nominal:
        return 'Cat-Nominal'
    else:
        return 'Other'

# Tính toán
missing_info = pd.DataFrame({
    'Feature': X_train.columns,
    'Feature Type': [get_feature_type(col) for col in X_train.columns],
    'Missing Count': X_train.isnull().sum().values,
    'Missing %': (X_train.isnull().sum() / len(X_train) * 100).values,
    'Present Count': X_train.notnull().sum().values,
    'Data Type': X_train.dtypes.values
})

# Phân loại theo mức độ missing
missing_info['Missing Level'] = pd.cut(
    missing_info['Missing %'],
    bins=[-0.1, 0, 5, 15, 100],
    labels=['None', 'Low (0-5%)', 'Medium (5-15%)', 'High (>15%)']
)

# Chỉ hiển thị features có missing
missing_only = missing_info[missing_info['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("\nFEATURES WITH MISSING VALUES:\n")
print(f"{'Feature':<30} | {'Type':<15} | {'Missing':<12} | {'%':<8} | {'Level':<15}")
print("-" * 95)
for idx, row in missing_only.iterrows():
    print(f"{row['Feature']:<30} | {row['Feature Type']:<15} | {row['Missing Count']:>10,} | {row['Missing %']:>6.2f}% | {row['Missing Level']:<15}")

# Overall statistics
print("\n" + "=" * 90)
print("OVERALL STATISTICS:")
print(f"   • Total observations: {len(X_train):,}")
print(f"   • Total features: {len(X_train.columns)}")
print(f"   • Features with missing: {len(missing_only)}")
print(f"   • Features without missing: {len(X_train.columns) - len(missing_only)}")
print(f"   • Total missing cells: {X_train.isnull().sum().sum():,}")
print(f"   • Overall missing rate: {(X_train.isnull().sum().sum() / (len(X_train) * len(X_train.columns)) * 100):.2f}%")
print("=" * 90)

DETAILED MISSING VALUES ANALYSIS

FEATURES WITH MISSING VALUES:

Feature                        | Type            | Missing      | %        | Level          
-----------------------------------------------------------------------------------------------
Previous Claims                | Numerical       |    364,029 |  30.34% | High (>15%)    
Occupation                     | Cat-Nominal     |    358,075 |  29.84% | High (>15%)    
Credit Score                   | Numerical       |    137,882 |  11.49% | Medium (5-15%) 
Number of Dependents           | Numerical       |    109,672 |   9.14% | Medium (5-15%) 
Customer Feedback              | Cat-Ordinal     |     77,824 |   6.49% | Medium (5-15%) 
Health Score                   | Numerical       |     74,076 |   6.17% | Medium (5-15%) 
Annual Income                  | Numerical       |     44,949 |   3.75% | Low (0-5%)     
Age                            | Numerical       |     18,705 |   1.56% | Low (0-5%)     
Marital Status            

In [121]:
# Step 1: Fill logic
# Number of Dependents: missing = không có người phụ thuộc
X_train['Number of Dependents'] = X_train['Number of Dependents'].fillna(0)
X_test['Number of Dependents'] = X_test['Number of Dependents'].fillna(0)

# Previous Claims: missing = khách hàng mới (chưa có claims)
X_train['Previous Claims'] = X_train['Previous Claims'].fillna(0)
X_test['Previous Claims'] = X_test['Previous Claims'].fillna(0)

# Occupation: tạo category 'Unknown'
X_train['Occupation'] = X_train['Occupation'].fillna('Unknown')
X_test['Occupation'] = X_test['Occupation'].fillna('Unknown')

In [122]:
# Step 2: Define feature groups
numeric_simple = ['Age', 'Annual Income', 'Health Score', 'Insurance Duration', 
                  'Vehicle Age', 'Number of Dependents', 'Previous Claims']
numeric_knn = ['Credit Score']

In [123]:
# Step 3: Create pipelines
# Numerical - Simple Median
numeric_simple_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Numerical - KNN (Credit Score)
numeric_knn_pipeline = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5, weights='distance')),
    ('scaler', StandardScaler())
])

# Categorical Ordinal
# ordinal_pipeline = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('encoder', OrdinalEncoder(
#         categories=[
#             ['High School', 'Bachelor', 'Master', 'PhD'],            # Education Level
#             ['Basic', 'Comprehensive', 'Premium'],                  # Policy Type  
#             ['Poor', 'Average', 'Good'],                            # Customer Feedback
#             ['Never', 'Monthly', 'Weekly', 'Daily']                 # Exercise Frequency
#         ],
#         handle_unknown='use_encoded_value',
#         unknown_value=-1
#     ))
# ])

# Categorical Nominal
nominal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])


In [124]:
# Step 4: Create ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num_simple', numeric_simple_pipeline, numeric_simple),
        ('num_knn', numeric_knn_pipeline, numeric_knn),
        ('cat_nominal', nominal_pipeline, categorical_nominal)
    ],
    remainder='passthrough', 
    verbose_feature_names_out=False
)

In [125]:
# Step 5: Fit and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nRESULTS:")
print(f"   • Original X_train shape: {X_train.shape}")
print(f"   • Processed X_train shape: {X_train_processed.shape}")
print(f"   • Processed X_test shape: {X_test_processed.shape}")
print(f"   • Missing values after: {np.isnan(X_train_processed).sum()}")

KeyboardInterrupt: 

In [ ]:
# Step 6: Get feature names (optional)

feature_names = preprocessor.get_feature_names_out()

print(f"\nFEATURE SUMMARY:")
print(f"   • Total features: {len(feature_names)}")
print(f"   • Numerical (simple): {len(numeric_simple)}")
print(f"   • Numerical (KNN): {len(numeric_knn)}")
print(f"   • Ordinal: {len(categorical_ordinal)}")
print(f"   • Nominal (one-hot): {len(categorical_nominal)}")

# Convert to DataFrame
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

## Outlier Processing

In [101]:
outlier_cols = [
    'Age', 'Annual Income', 'Number of Dependents',
    'Health Score', 'Previous Claims', 'Vehicle Age', 
    'Credit Score', 'Insurance Duration', 'Premium Amount'
]

print("=" * 90)
print("PART 1: OUTLIER ANALYSIS (IQR METHOD)")
print("=" * 90)

# Check which columns actually exist in the dataframe
available_cols = [c for c in outlier_cols if c in X_train.columns]
report_data = []

for col in available_cols:
    # 1. Calculate IQR
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # 2. Define Limits (for reporting only, not applied yet)
    lower_bound = max(Q1 - 1.5 * IQR, 0)
    upper_bound = Q3 + 1.5 * IQR
    
    # 3. Count actual outliers
    outliers_mask = (X_train[col] < lower_bound) | (X_train[col] > upper_bound)
    num_outliers = outliers_mask.sum()
    pct_outliers = (num_outliers / len(X_train)) * 100
    
    if num_outliers > 0:
        report_data.append({
            'Feature': col,
            'Count': num_outliers,
            'Percent': f"{pct_outliers:.2f}%",
            'Rec. Lower': f"{lower_bound:.2f}", # Recommended Lower Limit
            'Rec. Upper': f"{upper_bound:.2f}", # Recommended Upper Limit
            'Min': f"{X_train[col].min():.2f}",
            'Max': f"{X_train[col].max():.2f}"
        })

# 4. Display Report
if len(report_data) > 0:
    report_df = pd.DataFrame(report_data)
    print(f"Outliers detected in {len(report_data)} columns:\n")
    # Print formatted table
    print(report_df.to_string(index=False, justify='left'))
else:
    print("No significant outliers found using the IQR method.")

# --- OVERALL STATISTICS SECTION ---
total_outlier_cells = sum(item['Count'] for item in report_data)
total_features_analyzed = len(available_cols)
total_cells_analyzed = len(X_train) * total_features_analyzed

print("\n" + "=" * 90)
print("OVERALL OUTLIER STATISTICS:")
print(f"   • Total observations: {len(X_train):,}")
print(f"   • Features analyzed: {total_features_analyzed} (Numeric columns only)")
print(f"   • Features with outliers: {len(report_data)}")
print(f"   • Total outlier cells: {total_outlier_cells:,}")
if total_cells_analyzed > 0:
    print(f"   • Overall outlier rate: {(total_outlier_cells / total_cells_analyzed * 100):.2f}%")
else:
    print(f"   • Overall outlier rate: 0.00%")
print("=" * 90)

PART 1: OUTLIER ANALYSIS (IQR METHOD)
Outliers detected in 2 columns:

Feature          Count Percent Rec. Lower Rec. Upper Min  Max      
  Annual Income 67132  5.59%   0.00       99583.50   1.00 149997.00
Previous Claims 62066  5.17%   0.00           2.50   0.00      9.00

OVERALL OUTLIER STATISTICS:
   • Total observations: 1,200,000
   • Features analyzed: 8 (Numeric columns only)
   • Features with outliers: 2
   • Total outlier cells: 129,198
   • Overall outlier rate: 1.35%


In [ ]:
print("=" * 90)
print("PART 2: OUTLIER PROCESSING (APPLYING CAPPING)")
print("=" * 90)

processed_count = 0

for col in available_cols:
    # Recalculate limits (ensures consistency)
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = max(Q1 - 1.5 * IQR, 0)
    upper_bound = Q3 + 1.5 * IQR
    
    # Check for outliers before processing
    outliers_mask = (X_train[col] < lower_bound) | (X_train[col] > upper_bound)
    
    if outliers_mask.sum() > 0:
        # --- APPLY CAPPING ---
        # np.clip forces values < lower to lower and > upper to upper
        X_train[col] = np.clip(X_train[col], lower_bound, upper_bound)
        print(f"✓ Processed column: {col:<25} (Limits: {lower_bound:.2f} - {upper_bound:.2f})")
        processed_count += 1

if processed_count == 0:
    print("No columns required processing.")
else:
    print("-" * 90)
    print(f"Done! Applied capping to {processed_count} columns.")

## Categorical Encoder

In [ ]:
ordinal_features = ['Education Level', 'Policy Type', 'Customer Feedback', 'Exercise Frequency']
nominal_features = [col for col in categorical_features if col not in ordinal_features]

ordinal_education = ["High School", "Bachelor's", "Master's", "PhD"]
ordinal_policy = ["Basic", "Comprehensive", "Premium"]
ordinal_feedback = ["Poor", "Average", "Good"]
ordinal_exercise = ["Rarely", "Monthly", "Weekly", "Daily"]

In [ ]:
# ColumnTransformer để xử lý theo nhóm
preprocessor = ColumnTransformer(
    transformers=[
        # OneHot cho nominal categorical - ignore unknown values
        ("encode_cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features),   
        # Ordinal Encoder - use sentinel value for unknown
        ("encode_ord", OrdinalEncoder(categories=[ordinal_education, ordinal_policy, ordinal_feedback, ordinal_exercise],
             handle_unknown='use_encoded_value',
             unknown_value=-1 
         ), 
         ordinal_features)
    ],
    remainder="passthrough"
)

X_transformed = preprocessor.fit_transform(X_train, y_train)

In [ ]:
# Lấy tên các cột sau khi transform
feature_names = preprocessor.get_feature_names_out()

print("=" * 90)
print("TRANSFORMED FEATURE NAMES")
print("=" * 90)
print(f"Total features after transformation: {len(feature_names)}\n")

# Convert transformed data to DataFrame
X_train_transformed = pd.DataFrame(X_transformed, columns=feature_names, index=X_train.index)
X_test_transformed = preprocessor.transform(X_test)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

# Add target variable back to train data
train_processed = X_train_transformed.copy()
train_processed['Premium Amount'] = y_train

print(f"✓ Train data shape: {train_processed.shape}")
print(f"✓ Test data shape: {X_test_transformed.shape}")
print(f"\nFeature names preview:")
for i, name in enumerate(feature_names[:10], 1):
    print(f"   {i}. {name}")
if len(feature_names) > 10:
    print(f"   ... and {len(feature_names) - 10} more features")
print("=" * 90)

In [ ]:
# Save processed data to CSV
import os

# Create processed directory if not exists
os.makedirs("dataset/processed", exist_ok=True)

# Save train data (with target variable)
train_processed.to_csv("dataset/processed/train_processed.csv", index=False)
print(f"✓ Saved: dataset/processed/train_processed.csv ({train_processed.shape})")

# Save test data (without target variable)
X_test_transformed.to_csv("dataset/processed/test_processed.csv", index=False)
print(f"✓ Saved: dataset/processed/test_processed.csv ({X_test_transformed.shape})")

print("\n" + "=" * 90)
print("ALL FILES SAVED SUCCESSFULLY!")
print(f"   • Train: {train_processed.shape[0]:,} rows × {train_processed.shape[1]} columns")
print(f"   • Test:  {X_test_transformed.shape[0]:,} rows × {X_test_transformed.shape[1]} columns")
print("=" * 90)